In [ ]:
#Importation des bibliothèques
import os
import pandas as pd
import numpy as np
import tensorflow as tf
import warnings

warnings.filterwarnings('ignore')

I0000 00:00:1774448462.386166   18580 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1774448464.479307   18580 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774448467.758633   18580 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
#Chargement des données transformées
dossier_entree = 'data_transformed'

df_train = pd.read_csv(os.path.join(dossier_entree, 'train_transformed.csv'), sep=';')
df_val = pd.read_csv(os.path.join(dossier_entree, 'val_transformed.csv'), sep=';')
df_test = pd.read_csv(os.path.join(dossier_entree, 'test_transformed.csv'), sep=';')

cible = 'Default'

print(f"Chargement effectue. Dimensions du jeu d'entrainement : {df_train.shape}")

Chargement effectue. Dimensions du jeu d'entrainement : (178735, 32)


In [3]:
#Création du pipeline tf.data.Dataset
BATCH_SIZE = 256
AUTOTUNE = tf.data.AUTOTUNE

def creer_dataset(dataframe, cible, shuffle=False, batch_size=256):
    """
    Convertit un DataFrame pandas en tf.data.Dataset asynchrone et optimisé.
    """
    labels = dataframe[cible].astype('float32').values
    features = dataframe.drop(columns=[cible]).astype('float32').values
    
    ds = tf.data.Dataset.from_tensor_slices((features, labels))
    ds = ds.cache()
    
    if shuffle:
        ds = ds.shuffle(buffer_size=len(features), reshuffle_each_iteration=True)
        
    ds = ds.batch(batch_size)
    ds = ds.prefetch(buffer_size=AUTOTUNE)
    
    return ds

train_ds = creer_dataset(df_train, cible, shuffle=True, batch_size=BATCH_SIZE)
val_ds = creer_dataset(df_val, cible, shuffle=False, batch_size=BATCH_SIZE)
test_ds = creer_dataset(df_test, cible, shuffle=False, batch_size=BATCH_SIZE)

print("Pipelines tf.data.Dataset crees avec succes.")

E0000 00:00:1774448581.695490   18580 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1774448581.701690   19024 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1774448581.751146   18580 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Pipelines tf.data.Dataset crees avec succes.


In [4]:
#Procédure de validation et tests unitaires du pipeline
def valider_pipeline(dataset, batch_size_attendu, nb_features_attendu):
    """
    Test unitaire strict pour valider l'intégrité des tenseurs générés par le pipeline en isolation.
    """
    try:
        batches = list(dataset.take(1))
        assert len(batches) > 0, "Erreur critique : Le dataset est vide ou non itérable."
        
        x_batch, y_batch = batches[0]
        
        assert x_batch.shape[0] == batch_size_attendu, f"Erreur de dimension : Batch size {x_batch.shape[0]}, attendu {batch_size_attendu}"
        assert x_batch.shape[1] == nb_features_attendu, f"Erreur de dimension : Features {x_batch.shape[1]}, attendu {nb_features_attendu}"
        assert len(y_batch.shape) == 1, "Erreur structurelle : Les labels doivent être un vecteur 1D"
        
        assert x_batch.dtype == tf.float32, f"Erreur de type : Attendu tf.float32, obtenu {x_batch.dtype}"
        assert y_batch.dtype == tf.float32, f"Erreur de type : Attendu tf.float32, obtenu {y_batch.dtype}"
        
        assert not tf.math.reduce_any(tf.math.is_nan(x_batch)), "Erreur de valeur : Présence de NaN dans la matrice de caractéristiques"
        assert not tf.math.reduce_any(tf.math.is_nan(y_batch)), "Erreur de valeur : Présence de NaN dans le vecteur cible"
        
        # Vérification mathématique de la variance pour s'assurer que les données ne sont pas altérées en un vecteur uniforme
        variance_batch = tf.math.reduce_std(x_batch)
        assert variance_batch > 0.0, "Erreur mathématique : La variance du lot est nulle."
        
        print("Validation unitaire réussie. Pipeline opérationnel et stable.")
        print("Shape de la matrice de caractéristiques (X) :", x_batch.shape)
        print("Type de données (X) :", x_batch.dtype)
        
    except AssertionError as e:
        print(f"ECHEC DU TEST UNITAIRE : {e}")
        raise

nb_features = df_train.shape[1] - 1
valider_pipeline(train_ds, BATCH_SIZE, nb_features)

Validation unitaire réussie. Pipeline opérationnel et stable.
Shape de la matrice de caractéristiques (X) : (256, 31)
Type de données (X) : <dtype: 'float32'>


In [5]:
#Sauvegarde des graphes TensorFlow et validation du chargement
dossier_tf = 'data_tf_pipelines'
os.makedirs(dossier_tf, exist_ok=True)

# Sauvegarde au format natif tf.data
train_ds.save(os.path.join(dossier_tf, 'train'))
val_ds.save(os.path.join(dossier_tf, 'val'))
test_ds.save(os.path.join(dossier_tf, 'test'))

def valider_chargement_pipeline(chemin_acces, batch_size_attendu, nb_features_attendu):
    """
    Validation de la persistance des données sur le disque dur.
    """
    try:
        ds_charge = tf.data.Dataset.load(chemin_acces)
        element_spec = ds_charge.element_spec
        
        # Vérification de la signature des tenseurs rechargés
        assert element_spec[0].shape == (None, nb_features_attendu), "Erreur de dimension lors du rechargement des caractéristiques."
        assert element_spec[1].shape == (None,), "Erreur de dimension lors du rechargement de la cible."
        
        print(f"Validation du chargement réussie pour le répertoire : {chemin_acces}")
    except Exception as e:
        print(f"ECHEC DE LA VALIDATION DU CHARGEMENT : {e}")
        raise

valider_chargement_pipeline(os.path.join(dossier_tf, 'train'), BATCH_SIZE, nb_features)

Validation du chargement réussie pour le répertoire : data_tf_pipelines/train
